In [ ]:
import pandas as pd
df = pd.read_csv("/content/WA_Fn-UseC_-Telco-Customer-Churn.csv")
df.head()

In [ ]:
df.info()
df.describe()
df.shape

The dataset has 21 columns,7043 entries and out of 21 columns,18 are object type,2 are int,1 is float.columns of SeniorCitizen,tenure,MonthlyCharges are numeric and hence describe function calculates important metrics like count,mean,std max,min etc.shape function shows 7043 rows,21 columns.

In [ ]:
df.columns

In [ ]:
df.isnull().sum()
df['Churn'].value_counts()

The dataset is imbalanced towards No churn

In [ ]:
df[df['TotalCharges'] == " "]

In [ ]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df.isnull().sum()

In [17]:
# Drop missing rows (safe here, very few rows)
df = df.dropna()

In [18]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_style("whitegrid")

In [ ]:
sns.countplot(x='Churn', data=df)
plt.title("Churn Distribution")
plt.show()

In [ ]:
sns.histplot(data=df, x='tenure', hue='Churn', bins=30, kde=True)
plt.title("Tenure vs Churn")
plt.show()

In [ ]:
sns.histplot(data=df, x='MonthlyCharges', hue='Churn', bins=30, kde=True)
plt.title("Monthly Charges vs Churn")
plt.show()

In [ ]:
sns.countplot(x='Contract', hue='Churn', data=df)
plt.title("Contract Type vs Churn")
plt.show()

Customers with low tenure churn more.
Month-to-month contracts have highest churn.
Higher monthly charges increase churn.

In [23]:
df = df.drop('customerID', axis=1)

In [24]:
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

I converted the target variable into binary format to make it compatible with classification algorithms

In [25]:
binary_cols = df.select_dtypes(include='object').columns

for col in binary_cols:
    if df[col].nunique() == 2:
        df[col] = df[col].map({'Yes': 1, 'No': 0, 'Male':1, 'Female':0})

In [26]:
df = pd.get_dummies(df, drop_first=True)

In [ ]:
df.info()

In [28]:
X = df.drop('Churn', axis=1)
y = df['Churn']

In [ ]:
X.shape

In [30]:
def tenure_group(tenure):
    if tenure <= 12:
        return '0-1 Year'
    elif tenure <= 24:
        return '1-2 Years'
    elif tenure <= 48:
        return '2-4 Years'
    else:
        return '4+ Years'

df['tenure_group'] = df['tenure'].apply(tenure_group)

In [31]:
df['charges_per_tenure'] = df['MonthlyCharges'] / (df['tenure'] + 1)

In [32]:
df = pd.get_dummies(df, columns=['tenure_group'], drop_first=True)

In [33]:
X = df.drop('Churn', axis=1)
y = df['Churn']

In [ ]:
X.shape

In [35]:
from sklearn.model_selection import train_test_split

In [36]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [38]:
X_train.shape, X_test.shape

((5625, 34), (1407, 34))

In [39]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier

In [ ]:
lr = LogisticRegression(max_iter=1000)
lr.fit(X_train, y_train)

y_pred_lr = lr.predict(X_test)

In [41]:
rf = RandomForestClassifier()
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)

In [42]:
knn = KNeighborsClassifier()
knn.fit(X_train, y_train)

y_pred_knn = knn.predict(X_test)

In [43]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix

In [ ]:
def evaluate(y_test, y_pred, name):
    print(f"--- {name} ---")
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("Precision:", precision_score(y_test, y_pred))
    print("Recall:", recall_score(y_test, y_pred))
    print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
    print()

evaluate(y_test, y_pred_lr, "Logistic Regression")
evaluate(y_test, y_pred_rf, "Random Forest")
evaluate(y_test, y_pred_knn, "KNN")

In [45]:
from sklearn.model_selection import GridSearchCV

In [46]:
param_grid = {
    'C': [0.01, 0.1, 1, 10],
    'solver': ['liblinear', 'lbfgs']
}

In [ ]:
grid = GridSearchCV(
    LogisticRegression(max_iter=1000),
    param_grid,
    cv=5,
    scoring='recall'
)

grid.fit(X_train, y_train)

In [48]:
best_model = grid.best_estimator_

y_pred_best = best_model.predict(X_test)

In [ ]:
evaluate(y_test, y_pred_best, "Tuned Logistic Regression")

In [50]:
import pandas as pd

feature_importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': best_model.coef_[0]
})

feature_importance = feature_importance.sort_values(by='Importance', ascending=False)

feature_importance.head(10)

,Feature,Importance
11,InternetService_Fiber optic,0.803200
10,MultipleLines_Yes,0.318349
24,StreamingMovies_Yes,0.301984
22,StreamingTV_Yes,0.292250
6,PaperlessBilling,0.291092
1,SeniorCitizen,0.287365
33,tenure_group_4+ Years,0.264873
28,PaymentMethod_Electronic check,0.202046
9,MultipleLines_No phone service,0.158349
2,Partner,0.080204


In [51]:
feature_importance.tail(10)

,Feature,Importance
16,OnlineBackup_Yes,-0.109752
31,tenure_group_1-2 Years,-0.115751
29,PaymentMethod_Mailed check,-0.126299
27,PaymentMethod_Credit card (automatic),-0.147868
3,Dependents,-0.204753
20,TechSupport_Yes,-0.349975
14,OnlineSecurity_Yes,-0.392294
5,PhoneService,-0.556208
25,Contract_One year,-0.838957
26,Contract_Two year,-1.594382


Insight 1:

Customers with month-to-month contracts are significantly more likely to churn.

👉 Insight 2:

Customers with low tenure have higher churn probability.

👉 Insight 3:

Higher monthly charges are associated with increased churn.

👉 Insight 4:

Long-term contracts reduce churn risk.

The key drivers of churn were contract type, tenure, and monthly charges. Customers on month-to-month contracts with low tenure and higher charges showed higher churn probability